# Viral pipeline — visual overview

Charts for the dataset, model quality per source, and the content-model comparison. **Run All** to refresh after re-training.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
import joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, roc_auc_score

ML = Path.cwd()
if (ML / 'ml').exists():
    ML = ML / 'ml'            # running from repo root
elif ML.name == 'notebooks':
    ML = ML.parent            # running from ml/notebooks
sys.path.insert(0, str(ML))
from train.train_viral import TARGET, TEXT, split_indices

df = pd.read_parquet(ML / 'data' / 'train_dataset.parquet')
bundle = joblib.load(ML / 'models' / 'stage1_multisource.joblib')
y_all = df[TARGET].astype(int)
print(f'{len(df)} rows | viral rate {y_all.mean():.3f}')

## 1. Dataset by source

In [ ]:
g = df.groupby('source')[TARGET].agg(n='count', viral_rate='mean')
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
g['n'].plot.bar(ax=ax[0], color='#1e88e5', title='Posts per source')
g['viral_rate'].plot.bar(ax=ax[1], color='#43a047', title='Viral rate per source')
for a in ax: a.set_xlabel('')
plt.tight_layout(); plt.show()

## 2. Marketing-role distribution

In [ ]:
role_cols = sorted(c for c in df.columns if c.startswith('role_n_') and c != 'role_n_segments')
roles = pd.Series({c.replace('role_n_', ''): int(df[c].sum()) for c in role_cols}).sort_values()
roles.plot.barh(figsize=(8, 3.5), color='#8e24aa', title='Role distribution (segments)')
plt.tight_layout(); plt.show()

## 3. Model quality per source (test set)

In [ ]:
model, features = bundle['model'], bundle['features']
cm = bundle.get('content_model')
_, test_idx = split_indices(df, 0.2, 42)
test = df.iloc[test_idx].reset_index(drop=True)
X = test.reindex(columns=[c for c in features if c != 'content_score'], fill_value=0.0).astype(float)
if 'content_score' in features and cm is not None:
    X['content_score'] = cm.predict_proba(test[TEXT].astype(str))[:, 1]
X = X.reindex(columns=features, fill_value=0.0)
proba = model.predict_proba(X)[:, 1]
yt = test[TARGET].astype(int)

rows = []
for s in sorted(test['source'].dropna().unique()):
    m = (test['source'] == s).to_numpy()
    if yt[m].nunique() < 2:
        continue
    rows.append({'source': s, 'PR-AUC': average_precision_score(yt[m], proba[m]),
                 'ROC-AUC': roc_auc_score(yt[m], proba[m])})
q = pd.DataFrame(rows).set_index('source')
ax = q.plot.bar(figsize=(8, 3.5), title='Model quality per source')
ax.axhline(0.5, ls='--', c='grey'); ax.set_ylim(0, 1); ax.set_xlabel('')
plt.tight_layout(); plt.show()
q.round(3)

## 4. Content model: TF-IDF vs BERT (OOF PR-AUC)

In [ ]:
import json
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from features.text_content import build_content_model

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = cross_val_predict(build_content_model(), df[TEXT].fillna('').astype(str), y_all,
                        cv=cv, method='predict_proba')[:, 1]
tfidf_pr = average_precision_score(y_all, oof)
bert_pr = 0.428
bj = ML / 'data' / 'bert_content_metrics.json'
if bj.exists():
    bert_pr = json.loads(bj.read_text())['oof_pr_auc']
pd.Series({'TF-IDF': tfidf_pr, 'BERT (XLM-R)': bert_pr}).plot.bar(
    color=['#1e88e5', '#9e9e9e'], title='Content model PR-AUC (OOF)')
plt.ylim(0, 0.7); plt.tight_layout(); plt.show()

## Takeaways
- Model is strong on **YouTube** (most data), near-random on **X** (too little data).
- `content_score` is the dominant signal; role features mostly aid explainability.
- **TF-IDF beats BERT** at this data size — revisit BERT once data grows.
- Biggest lever: **crawl more X / Reddit data.**